In [1]:
# Fund Scorecard (0–100)
# composite = 30% × 3yr return rank + 25% × Sharpe rank + 20% × Alpha rank +
#              15% × expense ratio rank (inverse) + 10% × max DD rank (inverse)

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

_HERE = Path(__file__).resolve() if '__file__' in globals() else Path.cwd()

def _find_repo_root(start: Path) -> Path:
    cand = start
    for _ in range(15):
        if (cand / 'Data' / 'processed' / 'scheme_performance_clean.csv').exists():
            return cand
        if cand.name == 'notebooks':
            parent = cand.parent
            if (parent / 'Data' / 'processed' / 'scheme_performance_clean.csv').exists():
                return parent
        cand = cand.parent
    return start.parent

_REPO_ROOT = _find_repo_root(_HERE)
DATA_DIR = _REPO_ROOT / 'Data' / 'processed'

print('REPO_ROOT:', _REPO_ROOT.resolve())
print('DATA_DIR:', DATA_DIR.resolve())


REPO_ROOT: C:\Mutual Fund Analytics
DATA_DIR: C:\Mutual Fund Analytics\Data\processed


In [2]:
# --- Inputs ---
perf_path = DATA_DIR / 'scheme_performance_clean.csv'

if not perf_path.exists():
    raise FileNotFoundError(f'Missing: {perf_path.resolve()}')

perf_df = pd.read_csv(perf_path)

required_cols = {
    'amfi_code', 'scheme_name',
    'return_3yr_pct', 'sharpe_ratio', 'alpha',
    'expense_ratio_pct', 'max_drawdown_pct'
}
missing = required_cols - set(perf_df.columns)
if missing:
    raise ValueError(f'scheme_performance_clean.csv missing columns: {sorted(missing)}')

for c in ['return_3yr_pct', 'sharpe_ratio', 'alpha', 'expense_ratio_pct', 'max_drawdown_pct']:
    perf_df[c] = pd.to_numeric(perf_df[c], errors='coerce')

perf_df['amfi_code'] = pd.to_numeric(perf_df['amfi_code'], errors='coerce').astype('Int64')

print('Loaded rows:', len(perf_df))
print('Unique funds:', int(perf_df['amfi_code'].nunique()))

# Drop rows missing any scoring metric
score_df = perf_df.dropna(subset=list(required_cols)).copy()
print('Scoring rows after NaN-drop:', len(score_df))
print('Unique funds after NaN-drop:', int(score_df['amfi_code'].nunique()))


Loaded rows: 40
Unique funds: 40
Scoring rows after NaN-drop: 40
Unique funds after NaN-drop: 40


In [3]:
# --- Ranking helpers ---
# dense ranks: no gaps
def dense_rank(values: pd.Series, ascending: bool) -> pd.Series:
    return values.rank(method='dense', ascending=ascending).astype('Int64')

def rank_to_0_100_score(rank_1_based: pd.Series, n: int) -> pd.Series:
    # Best rank (1) => 100, worst rank (n) => 0
    if n <= 1:
        return pd.Series(100.0, index=rank_1_based.index)
    r0 = (rank_1_based.astype(float) - 1.0) / float(n - 1)
    return (100.0 * (1.0 - r0)).clip(0.0, 100.0)

# Higher-is-better metrics
score_df['rank_return_3yr'] = dense_rank(score_df['return_3yr_pct'], ascending=False)
score_df['rank_sharpe'] = dense_rank(score_df['sharpe_ratio'], ascending=False)
score_df['rank_alpha'] = dense_rank(score_df['alpha'], ascending=False)

# Inverse metrics (lower-is-better inputs)
score_df['rank_expense_inverse'] = dense_rank(score_df['expense_ratio_pct'], ascending=True)
score_df['rank_max_dd_inverse'] = dense_rank(score_df['max_drawdown_pct'], ascending=True)

N = int(max(
    score_df['rank_return_3yr'].max(),
    score_df['rank_sharpe'].max(),
    score_df['rank_alpha'].max(),
    score_df['rank_expense_inverse'].max(),
    score_df['rank_max_dd_inverse'].max()
))

score_df['score_return_3yr'] = rank_to_0_100_score(score_df['rank_return_3yr'], N)
score_df['score_sharpe'] = rank_to_0_100_score(score_df['rank_sharpe'], N)
score_df['score_alpha'] = rank_to_0_100_score(score_df['rank_alpha'], N)
score_df['score_expense_ratio'] = rank_to_0_100_score(score_df['rank_expense_inverse'], N)
score_df['score_max_drawdown'] = rank_to_0_100_score(score_df['rank_max_dd_inverse'], N)

score_df['fund_score_0_100'] = (
    0.30 * score_df['score_return_3yr'] +
    0.25 * score_df['score_sharpe'] +
    0.20 * score_df['score_alpha'] +
    0.15 * score_df['score_expense_ratio'] +
    0.10 * score_df['score_max_drawdown']
).clip(0.0, 100.0)

score_df = score_df.sort_values('fund_score_0_100', ascending=False).reset_index(drop=True)


In [4]:
# --- Display top funds ---
cols_to_show = [
    'amfi_code', 'scheme_name',
    'return_3yr_pct', 'score_return_3yr', 'rank_return_3yr',
    'sharpe_ratio', 'score_sharpe', 'rank_sharpe',
    'alpha', 'score_alpha', 'rank_alpha',
    'expense_ratio_pct', 'score_expense_ratio', 'rank_expense_inverse',
    'max_drawdown_pct', 'score_max_drawdown', 'rank_max_dd_inverse',
    'fund_score_0_100'
]
show_df = score_df[cols_to_show].head(20)
show_df


,amfi_code,scheme_name,return_3yr_pct,score_return_3yr,rank_return_3yr,sharpe_ratio,score_sharpe,rank_sharpe,alpha,score_alpha,rank_alpha,expense_ratio_pct,score_expense_ratio,rank_expense_inverse,max_drawdown_pct,score_max_drawdown,rank_max_dd_inverse,fund_score_0_100
0,119599,SBI Small Cap Fund - Direct Plan - Growth,23.14,97.435897,2,0.93,66.666667,14,1.13,51.282051,20,0.72,89.743590,5,-24.78,74.358974,11,77.051282
1,120842,Kotak Emerging Equity Fund - Regular - Growth,18.23,84.615385,7,0.96,74.358974,11,1.91,97.435897,2,1.56,35.897436,26,-21.92,61.538462,16,75.000000
2,101207,ABSL Small Cap Fund - Regular - Growth,22.38,94.871795,3,0.90,58.974359,17,1.84,92.307692,4,1.53,43.589744,23,-23.61,66.666667,14,74.871795
3,120843,Kotak Flexicap Fund - Regular - Growth,15.65,74.358974,11,0.98,76.923077,10,1.85,94.871795,3,1.45,53.846154,19,-19.50,51.282051,20,73.717949
4,119598,SBI Small Cap Fund - Regular Plan - Growth,23.39,100.000000,1,0.94,69.230769,13,1.23,56.410256,18,1.43,56.410256,18,-13.35,20.512821,32,69.102564
5,148567,Mirae Asset Large Cap Fund - Regular - Growth,14.81,58.974359,17,1.06,84.615385,7,1.62,76.923077,10,1.46,51.282051,20,-17.07,38.461538,25,65.769231
6,120505,ICICI Pru Midcap Fund - Regular - Growth,18.08,82.051282,8,0.95,71.794872,12,0.89,35.897436,26,1.36,66.666667,14,-21.84,56.410256,18,65.384615
7,148568,Mirae Asset Emerging Bluechip Fund - Regular -...,14.56,53.846154,19,0.91,61.538462,16,1.70,79.487179,9,1.52,46.153846,22,-33.15,97.435897,2,64.102564
8,100025,HDFC Short Term Debt Fund - Regular - Growth,7.37,10.256410,36,1.84,92.307692,4,1.98,100.000000,1,0.56,97.435897,2,-6.01,15.384615,34,62.307692
9,119094,Axis Midcap Fund - Regular - Growth,15.18,64.102564,15,0.80,41.025641,24,1.42,66.666667,14,1.38,64.102564,15,-32.38,94.871795,3,61.923077


In [5]:
# --- Save output ---
out_path = DATA_DIR / 'fund_scorecard.csv'
out_cols = [
    'amfi_code', 'scheme_name',
    'return_3yr_pct', 'rank_return_3yr', 'score_return_3yr',
    'sharpe_ratio', 'rank_sharpe', 'score_sharpe',
    'alpha', 'rank_alpha', 'score_alpha',
    'expense_ratio_pct', 'rank_expense_inverse', 'score_expense_ratio',
    'max_drawdown_pct', 'rank_max_dd_inverse', 'score_max_drawdown',
    'fund_score_0_100'
]

score_df[out_cols].to_csv(out_path, index=False)
print('Wrote:', out_path.resolve())
print('Top 5 composite:')
print(score_df[['amfi_code', 'scheme_name', 'fund_score_0_100']].head(5).to_string(index=False))


Wrote: C:\Mutual Fund Analytics\Data\processed\fund_scorecard.csv
Top 5 composite:
 amfi_code                                   scheme_name  fund_score_0_100
    119599     SBI Small Cap Fund - Direct Plan - Growth         77.051282
    120842 Kotak Emerging Equity Fund - Regular - Growth         75.000000
    101207        ABSL Small Cap Fund - Regular - Growth         74.871795
    120843        Kotak Flexicap Fund - Regular - Growth         73.717949
    119598    SBI Small Cap Fund - Regular Plan - Growth         69.102564
